In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="rl-applications",
    notebook_name="04_llm_alignment_case_study_experiments.ipynb"
)

# Experiments: LLM Alignment Case Study

This notebook produces runnable evidence for claims about the LLM alignment pipeline.

**Experiments:**
1. **DPO loss landscape** — verify that DPO loss decreases as the model increases the margin between chosen and rejected responses
2. **β sensitivity** — show that β controls the alignment tax: too low causes reward hacking, too high prevents learning
3. **Sycophancy detection** — demonstrate how a preference-trained model can become sycophantic and how to detect it

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")

---
## Experiment 1: DPO Loss Landscape

**Claim:** The DPO loss decreases as the model increases the log-probability ratio gap between chosen and rejected responses. The gradient pushes the model to increase P(chosen) and decrease P(rejected) relative to the reference.

**Why it matters in an interview:** Understanding the DPO loss landscape — how the gradient behaves, where it saturates, and how β controls the scale — is essential for explaining why DPO works and when it fails.

In [ ]:
# --- DPO loss as a function of the log-ratio margin ---

def sigmoid(x):
    """Numerically stable sigmoid."""
    return np.where(x >= 0, 1 / (1 + np.exp(-x)), np.exp(x) / (1 + np.exp(x)))

def dpo_loss(margin, beta):
    """
    DPO loss for a single preference pair.
    
    margin = log(pi(y_w|x)/pi_ref(y_w|x)) - log(pi(y_l|x)/pi_ref(y_l|x))
    
    L = -log(sigmoid(beta * margin))
    """
    return -np.log(sigmoid(beta * margin) + 1e-10)

def dpo_gradient(margin, beta):
    """
    Gradient of DPO loss w.r.t. margin.
    
    dL/d(margin) = -beta * (1 - sigmoid(beta * margin))
    """
    return -beta * (1 - sigmoid(beta * margin))

print("DPO loss and gradient functions defined.")
print("")
print("DPO loss formula:")
print("  L = -log(sigma(beta * margin))")
print("  where margin = log(pi(y_w)/pi_ref(y_w)) - log(pi(y_l)/pi_ref(y_l))")
print("")
print("Key insight: loss decreases as margin increases (model prefers chosen over rejected)")

In [ ]:
# --- Visualize loss landscape ---

margins = np.linspace(-6, 6, 200)
betas = [0.05, 0.1, 0.5, 1.0]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: DPO loss for different beta values
ax = axes[0]
colors = ['#f44336', '#ff9800', '#4caf50', '#2196f3']
for beta, color in zip(betas, colors):
    loss = dpo_loss(margins, beta)
    ax.plot(margins, loss, label='beta={}'.format(beta), color=color, linewidth=2)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Margin (log-ratio gap)')
ax.set_ylabel('DPO Loss')
ax.set_title('DPO Loss vs Margin')
ax.set_ylim(0, 5)
ax.legend()
ax.grid(True, alpha=0.3)

# Middle: Gradient magnitude
ax = axes[1]
for beta, color in zip(betas, colors):
    grad = dpo_gradient(margins, beta)
    ax.plot(margins, grad, label='beta={}'.format(beta), color=color, linewidth=2)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Margin (log-ratio gap)')
ax.set_ylabel('Gradient dL/d(margin)')
ax.set_title('DPO Gradient vs Margin')
ax.legend()
ax.grid(True, alpha=0.3)

# Right: Effective probability of choosing y_w
ax = axes[2]
beta_fixed = 0.1
prob_correct = sigmoid(beta_fixed * margins)
ax.plot(margins, prob_correct, 'b-', linewidth=2)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Random (50%)')
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.fill_between(margins, 0.5, prob_correct, where=prob_correct > 0.5, 
                alpha=0.2, color='green', label='Correct preference')
ax.fill_between(margins, prob_correct, 0.5, where=prob_correct < 0.5, 
                alpha=0.2, color='red', label='Wrong preference')
ax.set_xlabel('Margin (log-ratio gap)')
ax.set_ylabel('P(prefer chosen)')
ax.set_title('Implied Preference Probability (beta={})'.format(beta_fixed))
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print key values
print("DPO loss at key margins (beta=0.1):")
for m in [-3, -1, 0, 1, 3, 5]:
    loss_val = dpo_loss(m, 0.1)
    grad_val = dpo_gradient(m, 0.1)
    prob_val = sigmoid(0.1 * m)
    print("  margin={:+d}: loss={:.3f}, gradient={:.4f}, P(correct)={:.3f}".format(
        m, loss_val, grad_val, prob_val))

print("")
print("Key observations:")
print("  - Negative margin (model prefers rejected): high loss, strong gradient")
print("  - Zero margin (no preference): loss = log(2) = 0.693")
print("  - Positive margin (model prefers chosen): low loss, weak gradient")
print("  - Gradient saturates at large margins (learning stops for easy examples)")

### Result interpretation

**What the output shows:**
- DPO loss is a smooth function of the margin between chosen and rejected log-probability ratios
- Higher β makes the loss steeper — the model is penalized more harshly for wrong preferences but also saturates faster on correct ones
- The gradient is strongest when the model has the wrong preference (negative margin) and vanishes when the preference is already strongly correct
- This self-weighting is a key advantage: DPO focuses learning on examples the model currently gets wrong

**Interview one-liner:** "DPO's gradient is proportional to 1 - P(correct preference), so it automatically focuses on examples the model gets wrong and stops updating on already-correct examples."

---
## Experiment 2: β Sensitivity — The Alignment Tax

**Claim:** β controls the trade-off between reward optimization and staying close to the reference model. Too low β causes reward hacking (high reward but poor quality); too high β prevents meaningful learning.

**Why it matters in an interview:** β is the single most important hyperparameter in DPO. Understanding its effect on the Pareto frontier between reward and KL divergence is essential for tuning alignment pipelines.

In [ ]:
# --- Simulate DPO training at different beta values ---

class SimpleDPOSimulator:
    """
    Simulates DPO training on a simple preference task.
    
    The "model" is a set of logits for K responses.
    The "reference" is the initial logit distribution.
    Preference pairs are sampled from a true quality ordering.
    """
    def __init__(self, n_responses=10, seed=42):
        np.random.seed(seed)
        self.n = n_responses
        # True quality (hidden)
        self.true_quality = np.linspace(0, 1, n_responses)
        # Initial logits (reference model) - slightly noisy version of quality
        self.ref_logits = self.true_quality + np.random.randn(n_responses) * 0.3
        # Current logits (trainable)
        self.logits = self.ref_logits.copy()
    
    def sample_preference_pair(self):
        """Sample a preference pair from true quality."""
        i, j = np.random.choice(self.n, 2, replace=False)
        # Higher quality wins
        if self.true_quality[i] > self.true_quality[j]:
            return i, j  # i is chosen, j is rejected
        return j, i
    
    def compute_log_ratios(self):
        """Compute log(pi/pi_ref) for each response."""
        pi = np.exp(self.logits) / np.exp(self.logits).sum()
        pi_ref = np.exp(self.ref_logits) / np.exp(self.ref_logits).sum()
        return np.log(pi + 1e-10) - np.log(pi_ref + 1e-10)
    
    def compute_kl(self):
        """KL(pi || pi_ref)."""
        pi = np.exp(self.logits) / np.exp(self.logits).sum()
        pi_ref = np.exp(self.ref_logits) / np.exp(self.ref_logits).sum()
        return np.sum(pi * (np.log(pi + 1e-10) - np.log(pi_ref + 1e-10)))
    
    def compute_reward(self):
        """Expected true quality under current policy."""
        pi = np.exp(self.logits) / np.exp(self.logits).sum()
        return np.sum(pi * self.true_quality)
    
    def train_step(self, beta, lr=0.1):
        """One DPO gradient step."""
        w, l = self.sample_preference_pair()
        log_ratios = self.compute_log_ratios()
        margin = log_ratios[w] - log_ratios[l]
        
        # DPO gradient for the logits
        grad_weight = -beta * (1 - sigmoid(beta * margin))
        
        # Update: increase chosen logit, decrease rejected logit
        self.logits[w] -= lr * grad_weight  # grad_weight is negative, so this increases
        self.logits[l] += lr * grad_weight  # this decreases


print("DPO simulator defined.")
print("  - n_responses: number of response options")
print("  - true_quality: hidden ground truth (linearly spaced 0 to 1)")
print("  - Trains by sampling preference pairs and updating logits")

In [ ]:
# --- Run DPO at different beta values ---

betas_to_test = [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0]
n_steps = 500
n_seeds = 5

beta_results = {}

for beta in betas_to_test:
    rewards_all = []
    kls_all = []
    
    for seed in range(n_seeds):
        sim = SimpleDPOSimulator(n_responses=10, seed=seed)
        rewards = []
        kls = []
        
        for step in range(n_steps):
            sim.train_step(beta=beta, lr=0.05)
            rewards.append(sim.compute_reward())
            kls.append(sim.compute_kl())
        
        rewards_all.append(rewards)
        kls_all.append(kls)
    
    beta_results[beta] = {
        'rewards': np.array(rewards_all),
        'kls': np.array(kls_all),
    }

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_beta = plt.cm.viridis(np.linspace(0, 1, len(betas_to_test)))

# Left: Reward over training
ax = axes[0]
for beta, color in zip(betas_to_test, colors_beta):
    mean_r = beta_results[beta]['rewards'].mean(axis=0)
    ax.plot(mean_r, label='beta={}'.format(beta), color=color, linewidth=1.5)
ax.set_xlabel('Training step')
ax.set_ylabel('Expected true quality')
ax.set_title('Reward vs Training Steps')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

# Middle: KL over training
ax = axes[1]
for beta, color in zip(betas_to_test, colors_beta):
    mean_kl = beta_results[beta]['kls'].mean(axis=0)
    ax.plot(mean_kl, label='beta={}'.format(beta), color=color, linewidth=1.5)
ax.set_xlabel('Training step')
ax.set_ylabel('KL divergence from reference')
ax.set_title('KL Divergence vs Training Steps')
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)

# Right: Pareto frontier (reward vs KL at end of training)
ax = axes[2]
for beta, color in zip(betas_to_test, colors_beta):
    final_r = beta_results[beta]['rewards'][:, -1].mean()
    final_kl = beta_results[beta]['kls'][:, -1].mean()
    ax.scatter(final_kl, final_r, color=color, s=100, zorder=5)
    ax.annotate('b={}'.format(beta), (final_kl, final_r), 
                textcoords='offset points', xytext=(5, 5), fontsize=7)

# Connect points to show frontier
frontier_kl = [beta_results[b]['kls'][:, -1].mean() for b in betas_to_test]
frontier_r = [beta_results[b]['rewards'][:, -1].mean() for b in betas_to_test]
ax.plot(frontier_kl, frontier_r, 'k--', alpha=0.3)

ax.set_xlabel('KL divergence from reference')
ax.set_ylabel('Expected true quality')
ax.set_title('Pareto Frontier: Reward vs KL')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary table
print("Beta sensitivity summary (after {} steps):".format(n_steps))
print("{:<10} {:>15} {:>15} {:>15}".format(
    'Beta', 'Final Reward', 'Final KL', 'Reward/KL'))
print("-" * 55)
for beta in betas_to_test:
    r = beta_results[beta]['rewards'][:, -1].mean()
    kl = beta_results[beta]['kls'][:, -1].mean()
    ratio = r / (kl + 1e-6)
    print("{:<10} {:>15.4f} {:>15.4f} {:>15.4f}".format(beta, r, kl, ratio))

### Result interpretation

**What the output shows:**
- Low β (0.01-0.05): high reward but enormous KL divergence. The model deviates wildly from the reference — this is the reward hacking regime where the model exploits the preference signal without constraint
- High β (2.0-5.0): low KL (stays close to reference) but low reward. The constraint is too tight for meaningful learning
- Moderate β (0.1-0.5): best reward-per-unit-KL ratio. This is the sweet spot where the model learns preferences efficiently without excessive drift
- The Pareto frontier shows diminishing returns: the first units of KL buy a lot of reward improvement, but later units buy less and less

**Interview one-liner:** "β = 0.1 is a common starting point because it achieves most of the possible reward improvement with moderate KL divergence; the Pareto frontier shows diminishing returns — doubling KL rarely doubles reward."

---
## Experiment 3: Sycophancy Detection

**Claim:** When preference data is biased toward agreement (annotators preferred responses that agreed with the user's premise), the model learns to agree even when the user is wrong. This sycophancy can be detected by measuring agreement rate on factually incorrect prompts.

**Why it matters in an interview:** Sycophancy is one of the most discussed failure modes of aligned LLMs. Demonstrating how it arises from biased preference data — and how to detect and fix it — shows deep understanding of alignment challenges.

In [ ]:
# --- Sycophancy simulation ---

class SycophancySimulator:
    """
    Simulates preference learning where agreement is often preferred.
    
    The model learns a policy over [agree, disagree, hedging] responses.
    Prompts come in two types:
      - correct: user's premise is factually correct
      - incorrect: user's premise is factually wrong
    
    Biased preference data: agreement was preferred 80% of the time
    (because annotators liked polite, agreeing responses).
    """
    def __init__(self, agreement_bias=0.8, seed=42):
        np.random.seed(seed)
        self.agreement_bias = agreement_bias
        
        # Policy: [P(agree), P(disagree), P(hedge)] for correct and incorrect prompts
        # Reference model: roughly uniform
        self.ref_policy_correct = np.array([0.4, 0.3, 0.3])
        self.ref_policy_incorrect = np.array([0.3, 0.4, 0.3])
        
        # Trainable policy (starts from reference)
        self.policy_correct = self.ref_policy_correct.copy()
        self.policy_incorrect = self.ref_policy_incorrect.copy()
    
    def sample_preference(self, is_correct):
        """
        Sample a biased preference pair.
        
        Returns (chosen_action, rejected_action) where
        agreement is preferred with probability = agreement_bias.
        """
        actions = [0, 1, 2]  # agree, disagree, hedge
        chosen, rejected = np.random.choice(actions, 2, replace=False)
        
        # Bias: agreement wins most of the time
        if np.random.random() < self.agreement_bias:
            # Agreement preferred
            if chosen != 0:  # if chosen is not 'agree'
                chosen, rejected = rejected, chosen
            if chosen != 0:
                chosen = 0
                rejected = np.random.choice([1, 2])
        
        return chosen, rejected
    
    def train_step(self, beta=0.1, lr=0.01):
        """
        One DPO-like update step.
        """
        # Random prompt type
        is_correct = np.random.random() < 0.5
        chosen, rejected = self.sample_preference(is_correct)
        
        if is_correct:
            policy = self.policy_correct
            ref = self.ref_policy_correct
        else:
            policy = self.policy_incorrect
            ref = self.ref_policy_incorrect
        
        # Compute log-ratio margin
        log_ratio_w = np.log(policy[chosen] + 1e-10) - np.log(ref[chosen] + 1e-10)
        log_ratio_l = np.log(policy[rejected] + 1e-10) - np.log(ref[rejected] + 1e-10)
        margin = log_ratio_w - log_ratio_l
        
        # DPO gradient
        weight = beta * (1 - sigmoid(beta * margin))
        
        # Update policy
        policy[chosen] += lr * weight
        policy[rejected] -= lr * weight * 0.5
        
        # Normalize and clip
        policy[:] = np.clip(policy, 0.01, 0.98)
        policy[:] = policy / policy.sum()
    
    def get_metrics(self):
        """Return sycophancy metrics."""
        return {
            'agree_correct': self.policy_correct[0],
            'agree_incorrect': self.policy_incorrect[0],
            'disagree_incorrect': self.policy_incorrect[1],
        }


print("Sycophancy simulator defined.")
print("  Actions: [agree, disagree, hedge]")
print("  Prompt types: correct (user is right) and incorrect (user is wrong)")
print("  Bias: agreement preferred {}% of the time in preference data".format(
    int(0.8 * 100)))

In [ ]:
# --- Run sycophancy experiment with different bias levels ---

bias_levels = [0.5, 0.6, 0.7, 0.8, 0.9]
n_train_steps = 1000

sycophancy_results = {}

for bias in bias_levels:
    sim = SycophancySimulator(agreement_bias=bias, seed=42)
    
    agree_correct_curve = []
    agree_incorrect_curve = []
    disagree_incorrect_curve = []
    
    for step in range(n_train_steps):
        sim.train_step(beta=0.1, lr=0.01)
        metrics = sim.get_metrics()
        agree_correct_curve.append(metrics['agree_correct'])
        agree_incorrect_curve.append(metrics['agree_incorrect'])
        disagree_incorrect_curve.append(metrics['disagree_incorrect'])
    
    sycophancy_results[bias] = {
        'agree_correct': agree_correct_curve,
        'agree_incorrect': agree_incorrect_curve,
        'disagree_incorrect': disagree_incorrect_curve,
    }

# Plot results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_syc = plt.cm.Reds(np.linspace(0.3, 0.9, len(bias_levels)))

# Left: P(agree | user is CORRECT) - this SHOULD be high
ax = axes[0]
for bias, color in zip(bias_levels, colors_syc):
    ax.plot(sycophancy_results[bias]['agree_correct'], 
            label='bias={}'.format(bias), color=color, linewidth=1.5)
ax.set_xlabel('Training step')
ax.set_ylabel('P(agree)')
ax.set_title('P(agree | user CORRECT)\n(should be high)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# Middle: P(agree | user is INCORRECT) - this SHOULD be low
ax = axes[1]
for bias, color in zip(bias_levels, colors_syc):
    ax.plot(sycophancy_results[bias]['agree_incorrect'], 
            label='bias={}'.format(bias), color=color, linewidth=1.5)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
ax.set_xlabel('Training step')
ax.set_ylabel('P(agree)')
ax.set_title('P(agree | user INCORRECT)\n(should be LOW - sycophancy signal!)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# Right: Final sycophancy score vs bias level
ax = axes[2]
final_agree_incorrect = [sycophancy_results[b]['agree_incorrect'][-1] for b in bias_levels]
final_disagree_incorrect = [sycophancy_results[b]['disagree_incorrect'][-1] for b in bias_levels]

ax.bar(np.arange(len(bias_levels)) - 0.15, final_agree_incorrect, 0.3,
       label='P(agree | incorrect)', color='#f44336', alpha=0.8)
ax.bar(np.arange(len(bias_levels)) + 0.15, final_disagree_incorrect, 0.3,
       label='P(disagree | incorrect)', color='#4caf50', alpha=0.8)
ax.axhline(0.33, color='gray', linestyle='--', alpha=0.5, label='Uniform')
ax.set_xlabel('Agreement bias in preference data')
ax.set_ylabel('Final probability')
ax.set_title('Sycophancy vs Data Bias')
ax.set_xticks(range(len(bias_levels)))
ax.set_xticklabels([str(b) for b in bias_levels])
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print summary
print("Sycophancy analysis:")
print("{:<12} {:>20} {:>20} {:>15}".format(
    'Bias', 'P(agree|incorrect)', 'P(disagree|incorrect)', 'Sycophantic?'))
print("-" * 67)
for bias in bias_levels:
    agree = sycophancy_results[bias]['agree_incorrect'][-1]
    disagree = sycophancy_results[bias]['disagree_incorrect'][-1]
    is_syc = "YES" if agree > disagree else "No"
    print("{:<12} {:>20.3f} {:>20.3f} {:>15}".format(bias, agree, disagree, is_syc))

print("")
print("Detection rule: if P(agree | user is INCORRECT) > P(disagree | user is INCORRECT),")
print("the model is sycophantic. Fix by adding preference pairs where honest disagreement wins.")

### Result interpretation

**What the output shows:**
- As the agreement bias in preference data increases, P(agree | user is incorrect) rises above P(disagree | user is incorrect)
- At bias=0.5 (unbiased), the model correctly disagrees with incorrect premises
- At bias=0.8-0.9, the model becomes sycophantic: it agrees with the user even when the user is factually wrong
- The sycophancy is a direct consequence of the bias in preference data, not a model architecture issue

**Detection method:** Present the model with prompts containing known factual errors. If P(agree) > P(disagree) on these prompts, the model is sycophantic. Track this metric over training to catch sycophancy before deployment.

**Fix:** Add counter-examples to the preference data where the honest, corrective response is labeled as preferred over the agreeing response. Even 5-10% of such pairs can significantly reduce sycophancy.

**Interview one-liner:** "Sycophancy is a data problem, not a model problem — if annotators consistently preferred agreeing responses, the model learns to agree regardless of correctness. Detect by testing on known-incorrect prompts; fix by adding honesty-preferred pairs to the training data."

---
## Summary

| Claim | Evidence | Key number |
|-------|----------|------------|
| DPO gradient focuses on wrong predictions | Gradient ∝ 1 - P(correct), verified by loss landscape | Saturates at margin > 3 |
| β controls alignment tax | Low β → high reward + high KL; high β → low reward + low KL | β=0.1-0.3 is sweet spot |
| Pareto frontier shows diminishing returns | First units of KL buy more reward than later units | Reward/KL ratio peaks at moderate β |
| Biased preference data causes sycophancy | P(agree|incorrect) rises above P(disagree) at bias > 0.7 | Agreement bias directly maps to sycophancy |
| Sycophancy is detectable | Test on known-incorrect prompts, measure agreement rate | P(agree|incorrect) > 0.5 = sycophantic |

**For the full mathematical treatment:** see [llm-alignment-case-study-interview.md](./llm-alignment-case-study-interview.md)

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)